# Training the encoder
## Notes
I am testing three encoder architectures to learn the mapping from images to synthetic neural responses:
- A simple CNN (with batch norm, max pooling, dropout)
- A ResNet18 pretrained on ImageNet
- A simple CNN like the first **plus** skip connections

All of them expect a number of output neurons equal to the number of synthetic neurons (100) in the given dataset.

I am testing different learning rates and batch sizes. 
Optimiser is Adam, loss MSE
I am running 5-fold cross validation thorugh 40 epochs.
I am developing using Pytorch lightnning (especially Trainer, callbacks, LightningDataModule)and mlflow.

Also:
- mixed precision (with Pytorch Lightning - to reduce memory footprint during model training)


Let's look at the training of Resnet as an encoder.

In [1]:
import mlflow
from pathlib import Path


path_to_mlflow_runs = Path("/ceph/margrie/laura/neurodecoders/mlruns/")

# load all runs
mlflow.set_tracking_uri(f"file://{path_to_mlflow_runs}")
experiments = mlflow.search_experiments()
experiments

[<Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/351884261099482183', creation_time=1756523411390, experiment_id='351884261099482183', last_update_time=1756523411390, lifecycle_stage='active', name='dataset_sweep4', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/601020472894592046', creation_time=1756400093391, experiment_id='601020472894592046', last_update_time=1756400093391, lifecycle_stage='active', name='resnet_optimizer_scheduler_sweep2', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/261857491297849044', creation_time=1756323014084, experiment_id='261857491297849044', last_update_time=1756323014084, lifecycle_stage='active', name='fourth_run/resnet_encoder_comparison', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/325587642601194576', creation_time=1756317523977, experiment_id='325587642601194576', last_update_time=1

In [2]:
#  get the one with name fourth_run/resnet_encoder_comparison
name = "fourth_run/resnet_encoder_comparison"
experiment = [exp for exp in experiments if exp.name == name][0]
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.system/gpu_0_power_usage_watts,metrics.system/gpu_0_power_usage_percentage,metrics.system/system_memory_usage_megabytes,metrics.final_val_loss,...,params.dataset_input_shape,params.dataset_synthetic,params.best_checkpoint_path,params.dataset_output_neurons,params.dataset_total_size_mb,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.type
0,d51244febbe8491b85a58d6b564425d8,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:50:33.903000+00:00,2025-08-27 19:54:03.632000+00:00,4.3,1.9,14561.1,23.044058,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs16_epochs40_task24_fold_3,lporta,LOCAL
1,bb6f867e622949f8a6d65281a415c151,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:50:30.663000+00:00,2025-08-27 19:53:21.445000+00:00,7.9,3.4,25647.6,21.295284,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs32_epochs40_task25_fold_3,lporta,LOCAL
2,aa671e65f6df49d88785243dc7817709,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:48:29.150000+00:00,2025-08-27 19:49:52.860000+00:00,57.2,14.3,74475.0,19.075697,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs64_epochs40_task26_fold_3,lporta,LOCAL
3,5033243bec24428e96e0b24e484e2147,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:40.661000+00:00,2025-08-27 19:50:29.976000+00:00,7.8,3.4,21586.3,20.330101,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs32_epochs40_task25_fold_2,lporta,LOCAL
4,995c0ead11624d1ebd1928da42146c80,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:02.070000+00:00,2025-08-27 19:48:28.525000+00:00,57.4,14.3,72476.8,18.797646,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs64_epochs40_task26_fold_2,lporta,LOCAL
5,e2da836f328d4d5e815ff8974b17615e,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:02.023000+00:00,2025-08-27 19:50:33.211000+00:00,3.6,1.6,21594.2,23.925213,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs16_epochs40_task24_fold_2,lporta,LOCAL
6,c0251a8346cb4cc6ad69740d5f158cc6,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:46:26.938000+00:00,2025-08-27 19:49:05.122000+00:00,17.4,7.6,38902.4,18.227240,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr1e-3_bs64_epochs40_task23_fold_3,lporta,LOCAL
7,43690d5f06124941849c050166d29598,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:45:14.904000+00:00,2025-08-27 19:47:01.477000+00:00,56.9,14.2,70440.1,22.518211,...,"(10000, 1, 224, 224)",True,/ceph/margrie/laura/neurodecoders/workspace/ch...,100,1921.7681884765625,810cd81e821a31c270fba000ec2107625c46cf9f,neurodecoders/encoder/mlflow_training.py,lr5e-3_bs64_epochs40_task26_fold_1,lporta,LOCAL
8,

In [3]:
# let's only choose those that have status FINISHED and take the top 10 by val_loss and print train loss and val loss
runs = runs[runs["status"] == "FINISHED"]
runs = runs.sort_values(by=["metrics.val_loss"])
top_10_runs = runs.head(10)
for index, run in top_10_runs.iterrows():
    print(f"Train Loss: {run['metrics.train_loss']:2f}, Val Loss: {run['metrics.val_loss']:2f}, run_name: {run['tags.mlflow.runName']}")

Train Loss: 3.789138, Val Loss: 16.758606, run_name: lr5e-4_bs16_epochs40_task18_fold_2
Train Loss: 2.760064, Val Loss: 16.960333, run_name: lr1e-3_bs16_epochs40_task21_fold_2
Train Loss: 6.893279, Val Loss: 17.039955, run_name: lr5e-4_bs32_epochs40_task19_fold_2
Train Loss: 3.612371, Val Loss: 17.067032, run_name: lr1e-3_bs32_epochs40_task22_fold_2
Train Loss: 1.915294, Val Loss: 17.294289, run_name: lr1e-3_bs16_epochs40_task21_fold_3
Train Loss: 6.079440, Val Loss: 17.366009, run_name: lr5e-4_bs64_epochs40_task20_fold_3
Train Loss: 1.720755, Val Loss: 17.529062, run_name: lr5e-4_bs16_epochs40_task18_fold_3
Train Loss: 11.213946, Val Loss: 17.580606, run_name: lr5e-4_bs64_epochs40_task20_fold_2
Train Loss: 1.890170, Val Loss: 17.666798, run_name: lr1e-3_bs32_epochs40_task22_fold_3
Train Loss: 6.793271, Val Loss: 17.773151, run_name: lr1e-3_bs64_epochs40_task23_fold_2


After examining the results, I decided to move forward with pretrained ResNet18 with learning rate 1e-3 and batch size 32. I am going to move on with the following modifications:
- with a new dataset with higher spatial frequency components (perlin noise patterns)
- by separating training and test sets into different folders
- by testing:
    - learning rate schedulers (ReduceLROnPlateau, CosineAnnealingLR)
    - weight decay (1e-5, 1e-4)
    - different optimisers (AdamW, SGD with momentum)
    - warmup
- I should still be doing cross-validation (5 folds) to do hyperparameter search

A small note: I think I was doing CV wrong in the previous experiments. I was doing 3-fold CV but starting training on the next fold with the model weights from the previous fold. This is not correct, as each fold should be independent. I might need to re-run hyperparam search for learning dates ad batch size.

In [4]:
# Let's look at the optimiser and scheduler sweep experiment
# I've run this without cross validation and using lr=1e-3, batch_size=32, resnet encoder
name = "resnet_optimizer_scheduler_sweep2"

experiment = [exp for exp in experiments if exp.name == name][0]
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.system/gpu_0_power_usage_watts,metrics.system/gpu_0_power_usage_percentage,metrics.system/system_memory_usage_megabytes,metrics.final_val_loss,...,params.enable_mixed_precision,params.dataset_output_neurons,params.enable_early_stopping,params.dataset_sta_type,params.best_checkpoint_path,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.type
0,781fc52fcc2843c389dc33b03b851fed,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:09:47.780000+00:00,2025-08-28 17:09:48.235000+00:00,NaN,NaN,NaN,NaN,...,True,100,False,"gabor,70,70",None,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_sgd_sched_plateau,lporta,LOCAL
1,f1d23f0a7fda41b8b3ed56a1b5f0b68e,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:06:17.028000+00:00,2025-08-28 17:08:37.096000+00:00,173.5,75.4,39319.6,NaN,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_sgd_sched_cosine,lporta,LOCAL
2,1aa71ca6e841458a8a275183f755b4ea,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:06:04.154000+00:00,2025-08-28 17:08:27.028000+00:00,77.3,22.1,114035.0,inf,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_sgd_sched_step,lporta,LOCAL
3,13daf62b399b446f90b406841243fced,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:04:44.057000+00:00,2025-08-28 17:08:40.099000+00:00,6.7,2.9,38539.2,inf,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_sgd_sched_none,lporta,LOCAL
4,c037bdf02b6a40a393a3c6b5f796a3a7,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:04:29.106000+00:00,2025-08-28 17:04:29.568000+00:00,NaN,NaN,NaN,NaN,...,True,100,False,"gabor,70,70",None,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_adamw_sched_plateau,lporta,LOCAL
5,ab326e1d6b6a46698a90dfb58b69e40e,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:02:46.757000+00:00,2025-08-28 17:05:48.355000+00:00,77.6,22.2,123194.7,5.648138,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_adamw_sched_cosine,lporta,LOCAL
6,a5ec9fb98e654f2b9ea41b5f7914a112,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 17:01:04.765000+00:00,2025-08-28 17:05:59.459000+00:00,19.2,8.3,39517.2,5.728044,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_adamw_sched_step,lporta,LOCAL
7,e7d9a415f4474ccabf693c8a64f1743c,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 16:59:47.625000+00:00,2025-08-28 17:04:12.493000+00:00,6.3,2.7,38617.5,6.043922,...,True,100,False,"gabor,70,70",/ceph/margrie/laura/neurodecoders/workspace/ch...,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_adamw_sched_none,lporta,LOCAL
8,c983835d28ca486696525d6b821a55fe,601020472894592046,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-28 16:59:31.692000+00:00,2025-08-28 16:59:32.187000+00:00,NaN,NaN,NaN,NaN,...,True,100,False,"gabor,70,70",None,35a4c0384adbe8e7363f69ad527ae9bf0822fa10,neurodecoders/encoder/mlflow_training.py,opt_adam_sched_plateau,lporta,LOCAL
9,d2a3ffdc76d049c19bfeb686c19adac9,601020472894592046,FINISHED,

In [5]:
# let's only choose those that have status FINISHED and take the top 10 by val_loss and print train loss and val loss
runs = runs[runs["status"] == "FINISHED"]
runs = runs.sort_values(by=["metrics.val_loss"])
top_10_runs = runs.head(10)
for index, run in top_10_runs.iterrows():
    print(f"Train Loss: {run['metrics.train_loss']:2f}, Val Loss: {run['metrics.val_loss']:2f}, run_name: {run['tags.mlflow.runName']}")

Train Loss: 9.814347, Val Loss: 5.629071, run_name: opt_adam_sched_cosine
Train Loss: 9.909999, Val Loss: 5.648138, run_name: opt_adamw_sched_cosine
Train Loss: 9.483635, Val Loss: 5.709616, run_name: opt_adam_sched_step
Train Loss: 9.172269, Val Loss: 5.728044, run_name: opt_adamw_sched_step
Train Loss: 10.310371, Val Loss: 6.043922, run_name: opt_adamw_sched_none
Train Loss: 9.958501, Val Loss: 6.681193, run_name: opt_adam_sched_none
Train Loss: inf, Val Loss: inf, run_name: opt_sgd_sched_step
Train Loss: inf, Val Loss: inf, run_name: opt_sgd_sched_none
Train Loss: nan, Val Loss: nan, run_name: opt_sgd_sched_plateau
Train Loss: nan, Val Loss: nan, run_name: opt_sgd_sched_cosine


Interestingly, SGD didn't work well, we have infinite values for the train and validation losses. Also I don't see any difference between Adam and AdamW, and between no scheduler, ReduceLROnPlateau and CosineAnnealingLR.  

So for now I will move forward with Adam, no scheduler, lr=1e-3, batch_size=32, resnet encoder.
I will proceed exploring the impart of different filters for the simulated neurons in use.  

I am going to explore:
- perlin noise patterns (higher spatial frequency components)
- Gabor filters (to simulate V1-like neurons)
- periodic patterns (low frequency components)

In [6]:
# Let's look at the experiment regarding different datasets comparisons:
name = "dataset_sweep4"

experiment = [exp for exp in experiments if exp.name == name][0]
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.system/gpu_0_power_usage_watts,metrics.system/gpu_0_power_usage_percentage,metrics.system/system_memory_usage_megabytes,metrics.final_val_loss,...,params.optimizer_type,params.enable_mixed_precision,params.dataset_output_neurons,params.enable_early_stopping,params.dataset_sta_type,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.type
0,18f568dca2874054be0fc0d5ad74eda6,351884261099482183,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-30 03:10:19.815000+00:00,2025-08-30 03:18:02.476000+00:00,17.1,8.6,9369.7,4.853840,...,adam,True,100,False,"perlin_noise_patterns,70,70",b399e12f8ee36791cd297367f235b56b4d27c43c,neurodecoders/encoder/mlflow_training.py,"dataset=perlin_noise_patterns,70,70",lporta,LOCAL
1,a36d7ab406794274b677e6de4a77b8a4,351884261099482183,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-30 03:10:19.098000+00:00,2025-08-30 03:19:47.481000+00:00,12.8,6.4,10998.6,5.595065,...,adam,True,100,False,"gabor,70,70",b399e12f8ee36791cd297367f235b56b4d27c43c,neurodecoders/encoder/mlflow_training.py,"dataset=gabor,70,70",lporta,LOCAL
2,0b8167db91c94c7d961603547a69f31e,351884261099482183,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-30 03:10:17.259000+00:00,2025-08-30 03:27:24.932000+00:00,7.0,3.9,6203.6,31.267328,...,adam,True,100,False,"periodic_patterns,70,70",b399e12f8ee36791cd297367f235b56b4d27c43c,neurodecoders/encoder/mlflow_training.py,"dataset=periodic_patterns,70,70",lporta,LOCAL


In [7]:
runs = runs[runs["status"] == "FINISHED"]
runs = runs.sort_values(by=["metrics.val_loss"])
top_10_runs = runs.head(10)
for index, run in top_10_runs.iterrows():
    print(f"Train Loss: {run['metrics.train_loss']:2f}, Val Loss: {run['metrics.val_loss']:2f}, run_name: {run['tags.mlflow.runName']}")

Train Loss: 9.160123, Val Loss: 4.853840, run_name: dataset=perlin_noise_patterns,70,70
Train Loss: 9.203536, Val Loss: 5.595065, run_name: dataset=gabor,70,70
Train Loss: 26.899132, Val Loss: 31.267328, run_name: dataset=periodic_patterns,70,70


I notice data, if I compare "dataset_sweep4" and "resnet_optimizer_scheduler_sweep2" and look at the metric "mean_firing_rate_r2" that it seems it is learning well the firing rates in every single case.  

What I am finding surprising is another metric I am registering "best_classifier_accuracy_pred", which is the ability to classify cifar10 image categories from predicted firing rates, is significantly lower for these last two experiments (at best around .5). It was much higher in ""fourth_run/resnet_encoder_comparison" (max .95). My current explanation for that is the introduction of a key change in the codebase: creating separated files for train and test. I think there was a training leakage.  